In [2]:
from transformers import AutoConfig, AutoModel

config = AutoConfig.from_pretrained("/data/LLM-model/Qwen1.5-0.5B-Chat")
model = AutoModel.from_config(config)

display(config.base_model_tp_plan)
display(model)

{'layers.*.self_attn.q_proj': 'colwise',
 'layers.*.self_attn.k_proj': 'colwise',
 'layers.*.self_attn.v_proj': 'colwise',
 'layers.*.self_attn.o_proj': 'rowwise',
 'layers.*.mlp.gate_proj': 'colwise',
 'layers.*.mlp.up_proj': 'colwise',
 'layers.*.mlp.down_proj': 'rowwise'}

Qwen2Model(
  (embed_tokens): Embedding(151936, 1024)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=1024, out_features=2816, bias=False)
        (up_proj): Linear(in_features=1024, out_features=2816, bias=False)
        (down_proj): Linear(in_features=2816, out_features=1024, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((1024,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()

In [ ]:
import re
import torch.nn as nn
import tempfile
from transformers import AutoConfig, AutoModel
from vllm.config import ModelConfig
from vllm.distributed import (cleanup_dist_env_and_memory,
                              init_distributed_environment,
                              initialize_model_parallel)
from vllm.model_executor.layers.linear import ColumnParallelLinear, RowParallelLinear

temp_file = tempfile.mkstemp()[1]
init_distributed_environment(
    world_size=2,
    rank=0,
    distributed_init_method=f"file://{temp_file}",
    local_rank=0,
    backend="gloo",
)
initialize_model_parallel(2, 1)

MODEL_NAME = "/data/LLM-model/Qwen1.5-0.5B-Chat"

model_config = ModelConfig(
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    tokenizer_mode="auto",
    trust_remote_code=False,
    seed=0,
    dtype="bfloat16",
    revision=None,
    task="generate"
)


config = AutoConfig.from_pretrained("/data/LLM-model/Qwen1.5-0.5B-Chat")
model = AutoModel.from_config(config)


tp_plan = config.base_model_tp_plan

def replace_tp_linear_class(orig_module: nn.Linear, style: str):
    """
    In model configurations, we use a neutral type (string) to specify parallel
    styles, here we use it to translate nn.Linear into vllm-style tp Linear.
    """

    if not isinstance(style, str):
        raise ValueError(f"Unsupported parallel style type {type(style)}, expected str")

    input_size = orig_module.in_features
    output_size = orig_module.out_features
    bias = orig_module.bias is not None

    if style == "colwise":
        return ColumnParallelLinear(input_size, output_size, bias)
    elif style == "rowwise":
        return RowParallelLinear(input_size, output_size, bias)
    # We don't consider colwise_rep since it's used in lm_head
    else:
        raise ValueError(f"Unsupported parallel style value: {style}")


def tensor_parallelize(module: nn.Module, prefix: str =""):
    for child_name, child_module in module.named_children():
        qual_name = prefix + child_name
        for pattern, style in tp_plan.items():
            if re.match(pattern, qual_name) and isinstance(child_module, nn.Linear):
                new_module = replace_tp_linear_class(child_module, style)
                print(f"{qual_name}: {child_module} -> {new_module}")
                setattr(module, child_name, new_module)
        else:
            tensor_parallelize(child_module, prefix=f"{qual_name}.")

display(model)
tensor_parallelize(model)
display(model)

/home/c4rbon/miniconda3/envs/vllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/c4rbon/miniconda3/envs/vllm/lib/python3.10/site-packages/torch/distributed/distributed_c10d.py:335: UserWarning: Device capability of ccl unspecified, assuming `cpu` and `cuda`. Please specify it via the `devices` argument of `register_backend`.
  warnings.warn(


2024-12-18 15:22:08,864	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [10]:
import re

display(re.match('layers.*.self_attn.q_proj', "layers.0.self_attn.q_proj"))

<re.Match object; span=(0, 25), match='layers.0.self_attn.q_proj'>

In [ ]:
import 